# ATLAS Solar preprocessing tutorial

This notebook preprocesses downloaded climate data for one country and one variable.

It is structured as a tutorial for users who only need to edit the input parameters and then run the workflow step by step.

The workflow can:
1. read downloaded NetCDF files for each product;
2. standardise coordinates and CRS;
3. aggregate hourly data to daily data;
4. optionally merge ERA5-Land with ERA5 to fill missing coastal values;
5. crop the result around the selected country;
6. save final and intermediate outputs in a portable folder structure.


#### Documentation:

- `"era5"`: check "Variable name on CDS" https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation
- `"era5land"`: check "Varaible name on CDS" https://confluence.ecmwf.int/display/CKB/ERA5-Land%3A+data+documentation

## Step 1 — Import libraries

Run this cell first. If an import fails, install the missing package in your Python environment before continuing.


In [1]:
from pathlib import Path
import glob
import os

import geopandas as gpd
import numpy as np
import xarray as xr
import rioxarray  # required for the .rio accessor used to manage CRS information


## Step 2 — Input parameters

Edit only this cell for a new country, variable or product.

The download paths must point to the folders where the raw NetCDF files were downloaded.  
Use one path per product. For example, ERA5-Land and ERA5 can have different folders.

Final outputs are saved automatically in:

`../data/processed/{variable}/{country}/`

Intermediate support files ending in `_step1.nc` are saved in:

`../data/processed/{variable}/{country}/intermediate_step1_data/`


In [16]:
# Country name as written in the Natural Earth shapefile, using English country names.
# Examples: "Bolivia", "Peru", "Ecuador"
country = "Argentina"

# Variable folder name used in the download structure.
# Check the 1.1a) Notebook for the documentation
# Examples:
# "surface_solar_radiation_downwards"
# "total_cloud_cover"
# "2m_temperature"
variable = "surface_solar_radiation_downwards"

# Variable name inside the NetCDF files.
# For ERA5/ERA5-Land solar radiation this is usually "ssrd".
variable_short_name = "ssrd"

# Products to process.
# Use ["era5land", "era5"] when the final product should merge ERA5-Land and ERA5.
# Use a single product, for example ["era5"], for standalone preprocessing.
products = ["era5land",
            "era5"
           ]

# Raw download folders.
# Edit these paths if your downloaded files are stored elsewhere.
download_paths = {
    "era5land": Path(f"../data/cds_downloads/era5land/{variable}/{country.lower()}/"),
    "era5": Path(f"../data/cds_downloads/era5/{variable}/{country.lower()}/"),
}

# Country boundary file.
# This should point to the Natural Earth country shapefile.
shapefile_path = Path("../world_map/ne_50m_admin_0_countries.shp")

# Output folders.
output_path = Path(f"../data/processed/{variable}/{country.lower()}/")
intermediate_path = output_path / "intermediate_step1_data"

# Date labels used in output filenames.
# These labels do not subset the data by themselves; they document the expected period in the filename.
start_label = "1991-01"
end_label = "2020-12"

# Processing options.
overwrite = True
buffer_deg = 0.25
interpolation_method = "slinear"


## Step 3 — Utility functions

Run this section once. These cells define the functions used by the workflow.


In [17]:
def ensure_xarray_epsg4326(xdf):
    """Ensure that an xarray object has EPSG:4326 CRS and longitude/latitude spatial dimensions."""
    xdf = xdf.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)

    if xdf.rio.crs is None:
        xdf = xdf.rio.write_crs("EPSG:4326", inplace=False)

    return xdf


def roll_longitude_to_180(ds, lon_name="longitude"):
    """Convert longitudes from 0–360 degrees to -180–180 degrees."""
    ds = ds.assign_coords({
        lon_name: ((ds[lon_name] + 180) % 360) - 180
    })
    return ds.sortby(lon_name)


def fix_coords(xdf):
    """Standardise CRS and longitude/latitude coordinates before combining files."""
    xdf = ensure_xarray_epsg4326(xdf)
    xdf["longitude"] = np.round(xdf.longitude, 3)
    xdf["latitude"] = np.round(xdf.latitude, 3)

    if xdf.longitude.values.min() >= 0:
        xdf = roll_longitude_to_180(xdf)

    return xdf


### Compatibility note

The country geometry function has been updated to work with both older and newer GeoPandas versions.  
If you see a message such as `PROJ: proj_create_from_database`, it usually means the local conda environment is pointing to a missing PROJ data folder. The notebook now avoids the GeoPandas `union_all()` compatibility issue, which was the actual cause of the traceback shown above.

In [18]:
def get_country_geometry(country_name, shapefile):
    """Read the selected country geometry from the Natural Earth shapefile.

    This function is compatible with both recent and older GeoPandas versions.
    New GeoPandas versions provide ``union_all()``; older versions use
    ``unary_union``. The fallback avoids version-specific errors.
    """
    gdf = gpd.read_file(shapefile)

    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    else:
        gdf = gdf.to_crs(epsg=4326)

    matches = gdf[gdf["NAME_EN"].astype(str).str.lower() == country_name.lower()]

    if matches.empty:
        available_examples = ", ".join(sorted(gdf["NAME_EN"].dropna().unique())[:10])
        raise ValueError(
            f"Country '{country_name}' was not found in {shapefile}. "
            f"Check the English country name. Examples in the file include: {available_examples}"
        )

    if hasattr(matches.geometry, "union_all"):
        country_geometry = matches.geometry.union_all()
    else:
        country_geometry = matches.geometry.unary_union

    return gpd.GeoSeries([country_geometry], crs="EPSG:4326")


def cut_xdf(xdf, geometries, buffer_deg=0.0, buffer_cells=None):
    """Crop an xarray object using the bounding box of a country geometry."""
    xdf = xdf.sortby("latitude")

    lon_min, lat_min, lon_max, lat_max = geometries.total_bounds

    if buffer_cells is not None:
        if xdf.longitude.size < 2 or xdf.latitude.size < 2:
            raise ValueError("Cannot infer grid resolution from longitude/latitude coordinates.")

        dlon = float(np.abs(xdf.longitude.values[1] - xdf.longitude.values[0]))
        dlat = float(np.abs(xdf.latitude.values[1] - xdf.latitude.values[0]))

        lon_buffer = buffer_cells * dlon
        lat_buffer = buffer_cells * dlat
    else:
        lon_buffer = buffer_deg
        lat_buffer = buffer_deg

    return xdf.sel(
        longitude=slice(lon_min - lon_buffer, lon_max + lon_buffer),
        latitude=slice(lat_min - lat_buffer, lat_max + lat_buffer),
    )


In [19]:
def find_netcdf_files(data_path, pattern="*.nc"):
    """Return sorted NetCDF files from a download folder."""
    data_path = Path(data_path)
    files = sorted(data_path.glob(pattern))

    if not files:
        raise FileNotFoundError(f"No NetCDF files found in: {data_path}")

    return files


def load_data(data_path, pattern="*.nc"):
    """Load all NetCDF files in a product download folder."""
    files = find_netcdf_files(data_path, pattern=pattern)

    return xr.open_mfdataset(
        [str(file) for file in files],
        engine="netcdf4",
        preprocess=fix_coords,
        combine="by_coords",
    )


def aggregate_daily(xdf, var_name, product):
    """Aggregate an hourly product to daily values."""
    if "valid_time" in xdf.coords:
        xdf = xdf.rename({"valid_time": "time"})

    if var_name not in xdf:
        available = list(xdf.data_vars)
        raise KeyError(f"Variable '{var_name}' not found. Available variables: {available}")

    if product.lower() == "era5land" and var_name == "ssrd":
        # ERA5-Land ssrd is accumulated and the daily value is stored at hour 00.
        return xdf[var_name].sel(time=xdf.time.dt.hour == 0) / 24

    # Default behaviour for ERA5 and other products: daily mean.
    return xdf[var_name].resample(time="1D").mean()


In [20]:
def save_netcdf(xdf, output_filename, overwrite=True):
    """Save a Dataset or DataArray to NetCDF using float32 encoding."""
    output_filename = Path(output_filename)
    output_filename.parent.mkdir(parents=True, exist_ok=True)

    if output_filename.exists() and not overwrite:
        print(f"File already exists and overwrite is False: {output_filename}")
        return output_filename

    if "number" in xdf.coords:
        xdf = xdf.drop_vars("number")

    if "expver" in xdf.coords:
        xdf = xdf.drop_vars("expver")

    if hasattr(xdf, "name") and xdf.name is None:
        xdf.name = variable_short_name

    xdf = xdf.chunk({
        "time": 50,
        "latitude": 256,
        "longitude": 256,
    })

    if isinstance(xdf, xr.DataArray):
        encoding = {xdf.name: {"dtype": "float32"}}
    else:
        encoding = {var: {"dtype": "float32"} for var in xdf.data_vars}

    xdf.to_netcdf(
        output_filename,
        engine="netcdf4",
        encoding=encoding,
    )

    print(f"Data written to: {output_filename}")
    return output_filename


## Step 4 — Main processing functions

Use `process_era5land_era5_pair()` when you have both ERA5-Land and ERA5 and want to fill missing values.  
Use `process_single_product()` when you only need to preprocess one product.


In [21]:
def build_filename(product, var_name, start_label, end_label, suffix):
    """Create a standard output filename."""
    return f"{product}_{var_name}_{start_label}_{end_label}_{suffix}.nc"


def process_product_step1(product, data_path, var_name, start_label, end_label, intermediate_path, overwrite=True):
    """Load one product, aggregate it to daily scale and save the intermediate support file."""
    ds = load_data(data_path)
    daily = aggregate_daily(ds, var_name=var_name, product=product)

    intermediate_file = intermediate_path / build_filename(
        product=product,
        var_name=var_name,
        start_label=start_label,
        end_label=end_label,
        suffix="step1",
    )

    save_netcdf(daily, intermediate_file, overwrite=overwrite)
    return daily, intermediate_file


def process_era5land_era5_pair(
    download_paths,
    geometries,
    var_name,
    output_path,
    intermediate_path,
    start_label,
    end_label,
    overwrite=True,
    buffer_deg=0.25,
    interpolation_method="slinear",
):
    """Merge ERA5-Land and ERA5, crop to country and save the final processed file."""
    _, era5land_step1_file = process_product_step1(
        product="era5land",
        data_path=download_paths["era5land"],
        var_name=var_name,
        start_label=start_label,
        end_label=end_label,
        intermediate_path=intermediate_path,
        overwrite=overwrite,
    )

    _, era5_step1_file = process_product_step1(
        product="era5",
        data_path=download_paths["era5"],
        var_name=var_name,
        start_label=start_label,
        end_label=end_label,
        intermediate_path=intermediate_path,
        overwrite=overwrite,
    )

    era5land_step1 = xr.open_dataset(era5land_step1_file)
    era5_step1 = xr.open_dataset(era5_step1_file)

    era5_daily_interp = era5_step1.interp(
        latitude=era5land_step1.latitude.values,
        longitude=era5land_step1.longitude.values,
        method=interpolation_method,
    )

    merged = era5land_step1.combine_first(era5_daily_interp)
    cropped = cut_xdf(merged, geometries, buffer_deg=buffer_deg)

    final_file = output_path / f"{var_name}_{start_label}_{end_label}_processed.nc"
    save_netcdf(cropped, final_file, overwrite=overwrite)

    return cropped, final_file


def process_single_product(
    product,
    data_path,
    geometries,
    var_name,
    output_path,
    intermediate_path,
    start_label,
    end_label,
    overwrite=True,
    buffer_deg=0.25,
):
    """Preprocess one product only, crop to country and save the final processed file."""
    daily, _ = process_product_step1(
        product=product,
        data_path=data_path,
        var_name=var_name,
        start_label=start_label,
        end_label=end_label,
        intermediate_path=intermediate_path,
        overwrite=overwrite,
    )

    cropped = cut_xdf(daily, geometries, buffer_deg=buffer_deg)

    final_file = output_path / f"{product}_{var_name}_{start_label}_{end_label}_processed.nc"
    save_netcdf(cropped, final_file, overwrite=overwrite)

    return cropped, final_file

## Step 5 — Run preprocessing

Run this cell after checking the input parameters.

For the default solar radiation case, the notebook expects both ERA5-Land and ERA5 download paths and produces one merged final file.


In [22]:
# Create output folders.
output_path.mkdir(parents=True, exist_ok=True)
intermediate_path.mkdir(parents=True, exist_ok=True)

# Read the country boundary.
geometries = get_country_geometry(country, shapefile_path)

# Run the workflow.
if set(products) == {"era5land", "era5"}:
    processed_ds, final_file = process_era5land_era5_pair(
        download_paths=download_paths,
        geometries=geometries,
        var_name=variable_short_name,
        output_path=output_path,
        intermediate_path=intermediate_path,
        start_label=start_label,
        end_label=end_label,
        overwrite=overwrite,
        buffer_deg=buffer_deg,
        interpolation_method=interpolation_method,
    )
elif len(products) == 1:
    product = products[0]
    processed_ds, final_file = process_single_product(
        product=product,
        data_path=download_paths[product],
        geometries=geometries,
        var_name=variable_short_name,
        output_path=output_path,
        intermediate_path=intermediate_path,
        start_label=start_label,
        end_label=end_label,
        overwrite=overwrite,
        buffer_deg=buffer_deg,
    )
else:
    raise ValueError(
        "Unsupported product configuration. Use ['era5land', 'era5'] or a single product such as ['era5']."
    )

print(f"Final processed file: {final_file}")


Data written to: ../data/processed/total_cloud_cover/argentina/intermediate_step1_data/era5_tcc_1991-01_2020-12_step1.nc
Data written to: ../data/processed/total_cloud_cover/argentina/era5_tcc_1991-01_2020-12_processed.nc
Final processed file: ../data/processed/total_cloud_cover/argentina/era5_tcc_1991-01_2020-12_processed.nc


## Step 6 — Optional check

This cell only displays a compact summary of the processed dataset. It does not perform debugging and it does not create additional files.
